# STAG Single-Frame Pressure Classification with a Conv-SNN

This notebook is the only training and validation entry point. Each sample is one pressure map; the SNN time dimension is created internally by Poisson rate encoding.

## Important validation note

Following the selected experiment protocol, the official `test` split is used as validation at every epoch and selects `best_model.pt`. This introduces selection bias. The best validation score is therefore **not** an unbiased final-test estimate.

In [ ]:
from __future__ import annotations

import json
import os
import random
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from sklearn.metrics import confusion_matrix, f1_score
from spikingjelly.activation_based import functional
from torch import nn
from torch.utils.data import DataLoader, Subset

from data import create_single_frame_datasets
from model import SingleFrameConvSNN, count_trainable_parameters

# Change this value to "full" before a complete training run.
RUN_MODE = "smoke"
RUN_MODE = os.environ.get("SNN_RUN_MODE", RUN_MODE).strip().lower()
if RUN_MODE not in {"smoke", "full"}:
    raise ValueError("RUN_MODE must be 'smoke' or 'full'.")

SEED = 42
LEARNING_RATE = 1e-3
GAUSSIAN_NOISE_STD = 0.015
DROPOUT = 0.2
TAU = 2.0

if RUN_MODE == "smoke":
    EPOCHS = 1
    BATCH_SIZE = 4
    TIME_STEPS = 2
    SMOKE_TRAIN_SAMPLES = 4
    SMOKE_VALIDATION_SAMPLES = 4
    DEVICE = torch.device("cpu")
    EXPERIMENT_NAME = "single_frame_smoke"
else:
    EPOCHS = 200
    BATCH_SIZE = 32
    TIME_STEPS = 16
    DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    EXPERIMENT_NAME = "single_frame_full"

DATA_PATH = Path(
    os.environ.get(
        "STAG_DATA_PATH", "../stag_data/classification_lite.zip"
    )
).resolve()
OUTPUT_ROOT = Path(os.environ.get("SNN_OUTPUT_ROOT", "outputs")).resolve()
OUTPUT_DIR = OUTPUT_ROOT / EXPERIMENT_NAME
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALIDATION_WARNING = (
    "The official test split is used for validation at every epoch and "
    "for best-checkpoint selection; this introduces selection bias, so "
    "the best validation score is not an unbiased final-test estimate."
)

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
sns.set_theme(style="whitegrid")
print(f"Run mode: {RUN_MODE}")
print(f"Device: {DEVICE}")
print(f"Data: {DATA_PATH}")
print(f"Output: {OUTPUT_DIR}")
print(f"WARNING: {VALIDATION_WARNING}")

## 1. Load the official single-frame split

In [ ]:
metadata, splits, full_train_dataset, full_validation_dataset = (
    create_single_frame_datasets(DATA_PATH)
)

assert metadata.num_frames == 135_187
assert int(metadata.sensor_mask.sum()) == 548
assert splits.num_classes == 26
assert len(full_train_dataset) == 35_178
assert len(full_validation_dataset) == 15_522
assert "empty_hand" not in splits.class_names

split_summary = pd.DataFrame(
    {
        "split": ["train", "validation (official test)"],
        "samples": [len(full_train_dataset), len(full_validation_dataset)],
        "samples_per_class": [1_353, 597],
        "classes": [splits.num_classes, splits.num_classes],
    }
)
display(split_summary)

class_mapping = [
    {
        "label": label,
        "original_object_id": original_id,
        "name": splits.class_names[label],
    }
    for label, original_id in enumerate(splits.original_object_ids)
]
(OUTPUT_DIR / "class_mapping.json").write_text(
    json.dumps(class_mapping, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"Classes: {splits.class_names}")

## 2. Inspect single pressure maps

In [ ]:
sample_indices = np.linspace(
    0, len(full_train_dataset) - 1, num=6, dtype=int
)
fig, axes = plt.subplots(2, 3, figsize=(10, 7), constrained_layout=True)
for axis, dataset_index in zip(axes.ravel(), sample_indices):
    image, label = full_train_dataset[int(dataset_index)]
    view = axis.imshow(image[0], cmap="magma", vmin=0.0, vmax=1.0)
    axis.set_title(splits.class_names[int(label)])
    axis.set_axis_off()
fig.colorbar(view, ax=axes.ravel().tolist(), shrink=0.75, label="Normalized pressure")
fig.suptitle("Example STAG Single-Frame Pressure Maps", fontsize=14)
display(fig)
plt.close(fig)

## 3. Build DataLoaders

In [ ]:
if RUN_MODE == "smoke":
    train_indices = np.linspace(
        0,
        len(full_train_dataset) - 1,
        num=SMOKE_TRAIN_SAMPLES,
        dtype=int,
    ).tolist()
    validation_indices = np.linspace(
        0,
        len(full_validation_dataset) - 1,
        num=SMOKE_VALIDATION_SAMPLES,
        dtype=int,
    ).tolist()
    train_dataset = Subset(full_train_dataset, train_indices)
    validation_dataset = Subset(
        full_validation_dataset, validation_indices
    )
else:
    train_dataset = full_train_dataset
    validation_dataset = full_validation_dataset

generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=generator,
)
validation_loader = DataLoader(
    validation_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
)

images, labels = next(iter(train_loader))
assert images.ndim == 4 and tuple(images.shape[1:]) == (1, 32, 32)
assert labels.ndim == 1
assert torch.isfinite(images).all()
assert images.min() >= 0 and images.max() <= 1
invalid_mask = torch.from_numpy(~metadata.sensor_mask).unsqueeze(0)
assert torch.count_nonzero(images[:, :, invalid_mask[0]]) == 0
print(f"Used train samples: {len(train_dataset)}")
print(f"Used validation samples: {len(validation_dataset)}")
print(f"Batch shape: {tuple(images.shape)}")

## 4. Create the single-frame Conv-SNN

In [ ]:
model = SingleFrameConvSNN(
    num_classes=splits.num_classes,
    time_steps=TIME_STEPS,
    tau=TAU,
    dropout=DROPOUT,
).to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.CrossEntropyLoss()
sensor_mask_tensor = (
    torch.from_numpy(metadata.sensor_mask.astype(np.float32))
    .unsqueeze(0)
    .unsqueeze(0)
    .to(DEVICE)
)

model.eval()
set_seed(SEED)
try:
    with torch.no_grad():
        shape_check = model(images.to(DEVICE))
finally:
    functional.reset_net(model)
assert tuple(shape_check.shape) == (len(labels), splits.num_classes)
assert torch.isfinite(shape_check).all()
print(model)
print(f"Trainable parameters: {count_trainable_parameters(model):,}")
print(f"Output shape: {tuple(shape_check.shape)}")

## 5. Define training and validation loops

In [ ]:
def run_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: torch.device,
    num_classes: int,
    optimizer: torch.optim.Optimizer | None = None,
) -> dict[str, object]:
    is_training = optimizer is not None
    model.train(is_training)
    total_loss = 0.0
    sample_count = 0
    top1_correct = 0
    top3_correct = 0
    all_targets: list[torch.Tensor] = []
    all_predictions: list[torch.Tensor] = []

    for batch_images, batch_labels in loader:
        batch_images = batch_images.to(device)
        batch_labels = batch_labels.to(device)

        if is_training:
            noise = torch.randn_like(batch_images) * GAUSSIAN_NOISE_STD
            batch_images = (batch_images + noise).clamp(0.0, 1.0)
            batch_images = batch_images * sensor_mask_tensor
            optimizer.zero_grad(set_to_none=True)

        try:
            with torch.set_grad_enabled(is_training):
                logits = model(batch_images)
                loss = criterion(logits, batch_labels)
                if is_training:
                    loss.backward()
                    optimizer.step()
        finally:
            functional.reset_net(model)

        batch_size = batch_labels.numel()
        sample_count += batch_size
        total_loss += float(loss.detach()) * batch_size
        predictions = logits.detach().argmax(dim=1)
        top1_correct += int((predictions == batch_labels).sum())
        top3 = logits.detach().topk(k=3, dim=1).indices
        top3_correct += int(
            (top3 == batch_labels.unsqueeze(1)).any(dim=1).sum()
        )
        all_targets.append(batch_labels.detach().cpu())
        all_predictions.append(predictions.cpu())

    targets = torch.cat(all_targets).numpy()
    predictions = torch.cat(all_predictions).numpy()
    matrix = confusion_matrix(
        targets, predictions, labels=np.arange(num_classes)
    )
    macro_f1 = f1_score(
        targets,
        predictions,
        labels=np.arange(num_classes),
        average="macro",
        zero_division=0,
    )
    return {
        "loss": total_loss / sample_count,
        "top1": top1_correct / sample_count,
        "top3": top3_correct / sample_count,
        "macro_f1": float(macro_f1),
        "sample_count": sample_count,
        "confusion_matrix": matrix,
    }


def scalar_metrics(metrics: dict[str, object]) -> dict[str, float | int]:
    return {
        "loss": float(metrics["loss"]),
        "top1": float(metrics["top1"]),
        "top3": float(metrics["top3"]),
        "macro_f1": float(metrics["macro_f1"]),
        "sample_count": int(metrics["sample_count"]),
    }


def checkpoint_payload(
    epoch: int,
    metrics: dict[str, object],
) -> dict[str, object]:
    return {
        "epoch": int(epoch),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "model_config": model.get_config(),
        "class_names": list(splits.class_names),
        "validation_metrics": scalar_metrics(metrics),
        "validation_uses_official_test": True,
        "validation_warning": VALIDATION_WARNING,
    }


def is_better_validation(
    metrics: dict[str, object],
    best_top1: float,
    best_loss: float,
) -> bool:
    top1 = float(metrics["top1"])
    loss = float(metrics["loss"])
    return top1 > best_top1 or (
        np.isclose(top1, best_top1, rtol=0.0, atol=1e-12)
        and loss < best_loss
    )

## 6. Train and validate

Smoke mode runs one tiny real-data epoch. Full mode runs the configured 200 epochs.

In [ ]:
set_seed(SEED)
history: list[dict[str, float | int]] = []
best_epoch = -1
best_top1 = float("-inf")
best_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(
        model,
        train_loader,
        criterion,
        DEVICE,
        splits.num_classes,
        optimizer=optimizer,
    )
    validation_metrics = run_epoch(
        model,
        validation_loader,
        criterion,
        DEVICE,
        splits.num_classes,
    )

    row: dict[str, float | int] = {"epoch": epoch}
    for prefix, metrics in (
        ("train", train_metrics),
        ("validation", validation_metrics),
    ):
        for name, value in scalar_metrics(metrics).items():
            row[f"{prefix}_{name}"] = value
    history.append(row)

    if is_better_validation(validation_metrics, best_top1, best_loss):
        best_epoch = epoch
        best_top1 = float(validation_metrics["top1"])
        best_loss = float(validation_metrics["loss"])
        torch.save(
            checkpoint_payload(epoch, validation_metrics),
            OUTPUT_DIR / "best_model.pt",
        )

    print(
        f"Epoch {epoch:03d}/{EPOCHS:03d} | "
        f"train loss={train_metrics['loss']:.4f}, "
        f"top1={train_metrics['top1']:.3f} | "
        f"validation loss={validation_metrics['loss']:.4f}, "
        f"top1={validation_metrics['top1']:.3f}, "
        f"top3={validation_metrics['top3']:.3f}"
    )

torch.save(
    checkpoint_payload(EPOCHS, validation_metrics),
    OUTPUT_DIR / "last_model.pt",
)
history_frame = pd.DataFrame(history)
history_frame.to_csv(OUTPUT_DIR / "history.csv", index=False)
assert (OUTPUT_DIR / "best_model.pt").is_file()
assert (OUTPUT_DIR / "last_model.pt").is_file()
display(history_frame)

## 7. Load the best checkpoint and visualize results

In [ ]:
best_checkpoint = torch.load(
    OUTPUT_DIR / "best_model.pt",
    map_location=DEVICE,
    weights_only=False,
)
model.load_state_dict(best_checkpoint["model_state_dict"])
set_seed(SEED)
best_validation_metrics = run_epoch(
    model,
    validation_loader,
    criterion,
    DEVICE,
    splits.num_classes,
)
print(f"Selected epoch: {best_checkpoint['epoch']}")
print(f"Re-evaluated validation metrics: {scalar_metrics(best_validation_metrics)}")
print(f"WARNING: {VALIDATION_WARNING}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8), constrained_layout=True)
plots = (
    ("loss", "Cross-Entropy Loss", "Loss"),
    ("top1", "Top-1 Accuracy", "Accuracy"),
    ("top3", "Top-3 Accuracy", "Accuracy"),
    ("macro_f1", "Macro F1", "Score"),
)
for axis, (column, title, ylabel) in zip(axes.ravel(), plots):
    axis.plot(
        history_frame["epoch"],
        history_frame[f"train_{column}"],
        marker="o",
        label="Train",
    )
    axis.plot(
        history_frame["epoch"],
        history_frame[f"validation_{column}"],
        marker="o",
        label="Validation (Official Test)",
    )
    axis.set_title(title)
    axis.set_xlabel("Epoch")
    axis.set_ylabel(ylabel)
    axis.legend()
fig.suptitle("Single-Frame Conv-SNN Training History", fontsize=15)
fig.savefig(OUTPUT_DIR / "training_curves.png", dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)

In [ ]:
matrix = np.asarray(best_validation_metrics["confusion_matrix"])
fig, axis = plt.subplots(figsize=(14, 12), constrained_layout=True)
sns.heatmap(
    matrix,
    cmap="Blues",
    xticklabels=splits.class_names,
    yticklabels=splits.class_names,
    cbar_kws={"label": "Frame count"},
    ax=axis,
)
axis.set_title("Validation Confusion Matrix (Official Test Split)")
axis.set_xlabel("Predicted object")
axis.set_ylabel("True object")
axis.tick_params(axis="x", rotation=90)
axis.tick_params(axis="y", rotation=0)
fig.savefig(OUTPUT_DIR / "confusion_matrix.png", dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)

In [ ]:
row_totals = matrix.sum(axis=1)
class_accuracy = np.divide(
    np.diag(matrix),
    row_totals,
    out=np.zeros(splits.num_classes, dtype=np.float64),
    where=row_totals > 0,
)
fig, axis = plt.subplots(figsize=(14, 6), constrained_layout=True)
axis.bar(splits.class_names, class_accuracy, color="#4472C4")
axis.set_title("Per-Class Validation Accuracy (Official Test Split)")
axis.set_xlabel("Object class")
axis.set_ylabel("Accuracy")
axis.set_ylim(0.0, 1.0)
axis.tick_params(axis="x", rotation=90)
fig.savefig(OUTPUT_DIR / "class_accuracy.png", dpi=180, bbox_inches="tight")
display(fig)
plt.close(fig)

## 8. Save the experiment summary

In [ ]:
summary = {
    "run_mode": RUN_MODE,
    "num_frames": metadata.num_frames,
    "sensor_count": int(metadata.sensor_mask.sum()),
    "num_classes": splits.num_classes,
    "class_names": list(splits.class_names),
    "full_train_samples": len(full_train_dataset),
    "full_validation_samples": len(full_validation_dataset),
    "used_train_samples": len(train_dataset),
    "used_validation_samples": len(validation_dataset),
    "epochs": EPOCHS,
    "best_epoch": int(best_checkpoint["epoch"]),
    "selected_validation_metrics": best_checkpoint["validation_metrics"],
    "reevaluated_validation_metrics": scalar_metrics(
        best_validation_metrics
    ),
    "model_config": model.get_config(),
    "validation_uses_official_test": True,
    "validation_warning": VALIDATION_WARNING,
    "output_files": [
        "best_model.pt",
        "last_model.pt",
        "history.csv",
        "summary.json",
        "class_mapping.json",
        "training_curves.png",
        "confusion_matrix.png",
        "class_accuracy.png",
    ],
}
(OUTPUT_DIR / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

expected_outputs = [OUTPUT_DIR / name for name in summary["output_files"]]
missing_outputs = [path.name for path in expected_outputs if not path.is_file()]
if missing_outputs:
    raise AssertionError(f"Missing outputs: {missing_outputs}")

display(pd.DataFrame([summary["reevaluated_validation_metrics"]]))
print(f"Saved experiment outputs to: {OUTPUT_DIR}")
print(f"WARNING: {VALIDATION_WARNING}")

## Interpretation

This is a lightweight Conv-SNN baseline using the Nature paper's official single-frame split and balancing flags, with `empty_hand` removed. It is not an exact reproduction of the paper's modified ResNet or its 27-class score. Because the official test split selects the best checkpoint, report the resulting score as validation performance with selection bias.